In [17]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


from sklearn.metrics import accuracy_score, mean_squared_error, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

from sklearn.base import clone

import xgboost as xgb


In [42]:
X_train = pd.read_csv('data/X_train.csv',index_col='ROW_ID')
X_test = pd.read_csv('data/X_test.csv',index_col='ROW_ID').fillna(0)

y_train = pd.read_csv('data/y_train.csv',index_col='ROW_ID')
sample_submission = pd.read_csv('data/sample_submission.csv',index_col='ROW_ID')

# Features

In [43]:
RET_features = [f'RET_{i}' for i in range(1,20)]
SIGNED_VOLUME_features = [f'SIGNED_VOLUME_{i}' for i in range(1,20)]
TURNOVER_features = ['AVG_DAILY_TURNOVER']

### Average perfs

In [44]:
for i in [3,5,10,15,20]:
    X_train[ f'AVERAGE_PERF_{i}'] = X_train[RET_features[:i+1]].mean(1)
    X_train[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_train.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')
    
    X_test[ f'AVERAGE_PERF_{i}'] = X_test[RET_features[:i+1]].mean(1)
    X_test[ f'ALLOCATIONS_AVERAGE_PERF_{i}'] = X_test.groupby('TS')[ f'AVERAGE_PERF_{i}'].transform('mean')

### EMAs

In [45]:
ret_cols = [f"RET_{i}" for i in range(1, 21)]
r = X_train[ret_cols].to_numpy()

rtest = X_test[ret_cols].to_numpy()

def ema(arr, L):
    alpha = 2/(L+1)
    w = (1-alpha) ** np.arange(L)  # 0..L-1
    w = w / w.sum()
    return (arr[:, :L] * w).sum(axis=1)

X_train["ema3"]  = ema(r, 3)
X_train["ema5"]  = ema(r, 5)
X_train["ema10"] = ema(r,10)


X_test["ema3"]  = ema(rtest, 3)
X_test["ema5"]  = ema(rtest, 5)
X_test["ema10"] = ema(rtest,10)

m20 = r.mean(axis=1)
s20 = r.std(axis=1, ddof=0)

m20test = rtest.mean(axis=1)
s20test = rtest.std(axis=1, ddof=0)

X_train["z20"] = m20 / (s20 + 1e-12)
X_test["z20"] = m20test / (s20test + 1e-12)

### Sg streak et entropy

In [46]:
sign = np.sign(r)  
signtest = np.sign(rtest)

def last_streak_len(sig_row, positive=True):
    # part de RET_1 vers RET_20
    target = 1 if positive else -1
    cnt = 0
    for v in sig_row[:20]:  # [:20] explicite
        if v == target:
            cnt += 1
        else:
            break
    return cnt

X_train["streak_pos"] = [last_streak_len(s, True) for s in sign]
X_train["streak_neg"] = [last_streak_len(s, False) for s in sign]

X_test["streak_pos"] = [last_streak_len(s, True) for s in signtest]
X_test["streak_neg"] = [last_streak_len(s, False) for s in signtest]

p = (r > 0).mean(axis=1)
p = np.clip(p, 1e-9, 1 - 1e-9)
X_train["sign_entropy20"] = -(p*np.log(p) + (1-p)*np.log(1-p))

ptest = (rtest > 0).mean(axis=1)    
ptest = np.clip(ptest, 1e-9, 1 - 1e-9)
X_test["sign_entropy20"] = -(ptest*np.log(ptest) + (1-ptest)*np.log(1-ptest))


### Rendement (sharpe10 / sortino10 / t-stat)

In [56]:
r10 = r[:, :10]
m10 = r10.mean(axis=1)
s10 = r10.std(axis=1, ddof=0)
neg10 = np.minimum(r10, 0)

r10test = rtest[:, :10]
m10test = r10test.mean(axis=1)
s10test = r10test.std(axis=1, ddof=0)
neg10test = np.minimum(r10test, 0)

X_train["sharpe10"]   = m10 / (s10 + 1e-12)
downside10 = np.sqrt((neg10**2).mean(axis=1))
X_train["sortino10"]  = m10 / (downside10 + 1e-12)

X_test["sharpe10"]   = m10test / (s10test + 1e-12)
downside10test = np.sqrt((neg10test**2).mean(axis=1))
X_test["sortino10"]  = m10test / (downside10test + 1e-12)


Max drawdown, recovery

In [57]:
X_train["vol10"]         = s10
X_train["downside_dev10"]= downside10

X_test["vol10"]         = s10test
X_test["downside_dev10"]= downside10test

def maxdd_and_recovery(row):
    c = np.cumsum(row)                   # equity curve 20j
    peak = np.maximum.accumulate(c)
    dd = (c - peak)
    maxdd = dd.min()                     # drawdown (négatif)
    # recovery: jours depuis le dernier pic
    last_peak_idx = np.where(c == peak)[0][-1]
    recovery = 20 - 1 - last_peak_idx
    return maxdd, recovery

md_rec = np.apply_along_axis(maxdd_and_recovery, 1, r)
X_train["maxdd20"]   = md_rec[:,0]
X_train["recovery20"]= md_rec[:,1]

md_rec_test = np.apply_along_axis(maxdd_and_recovery, 1, rtest)
X_test["maxdd20"]    = md_rec_test[:,0]
X_test["recovery20"] = md_rec_test[:,1]


In [58]:
features = features = [col for col in X_train.columns if col not in ['TS', 'ALLOCATION']]

In [47]:
print(len(features), features)

58 ['RET_20', 'RET_19', 'RET_18', 'RET_17', 'RET_16', 'RET_15', 'RET_14', 'RET_13', 'RET_12', 'RET_11', 'RET_10', 'RET_9', 'RET_8', 'RET_7', 'RET_6', 'RET_5', 'RET_4', 'RET_3', 'RET_2', 'RET_1', 'SIGNED_VOLUME_20', 'SIGNED_VOLUME_19', 'SIGNED_VOLUME_18', 'SIGNED_VOLUME_17', 'SIGNED_VOLUME_16', 'SIGNED_VOLUME_15', 'SIGNED_VOLUME_14', 'SIGNED_VOLUME_13', 'SIGNED_VOLUME_12', 'SIGNED_VOLUME_11', 'SIGNED_VOLUME_10', 'SIGNED_VOLUME_9', 'SIGNED_VOLUME_8', 'SIGNED_VOLUME_7', 'SIGNED_VOLUME_6', 'SIGNED_VOLUME_5', 'SIGNED_VOLUME_4', 'SIGNED_VOLUME_3', 'SIGNED_VOLUME_2', 'SIGNED_VOLUME_1', 'AVG_DAILY_TURNOVER', 'AVERAGE_PERF_3', 'ALLOCATIONS_AVERAGE_PERF_3', 'AVERAGE_PERF_5', 'ALLOCATIONS_AVERAGE_PERF_5', 'AVERAGE_PERF_10', 'ALLOCATIONS_AVERAGE_PERF_10', 'AVERAGE_PERF_15', 'ALLOCATIONS_AVERAGE_PERF_15', 'AVERAGE_PERF_20', 'ALLOCATIONS_AVERAGE_PERF_20', 'ema3', 'ema5', 'ema10', 'z20', 'streak_pos', 'streak_neg', 'sign_entropy20']


In [54]:
# highlight any super high value in X columns
for col in features:
    tmp = X_train[col].quantile(0.99)
    if tmp > 1000:
        print(col)
        X_train.loc[X_train[col] > tmp, col] = X_train[col].median()


## ridge

In [59]:
np.linalg.cond(X_train[features].to_numpy(dtype=float))

np.float64(1.4440016203401402e+27)

In [90]:
def clean_fold(X_tr, X_val):
    # 1) forcer numérique et virer inf/NaN
    X_tr = X_tr.replace([np.inf, -np.inf], np.nan).astype(np.float64)
    X_val = X_val.replace([np.inf, -np.inf], np.nan).astype(np.float64)

    # 2) retirer colonnes constantes (std=0) sur TRAIN (et les mêmes côté VAL)
    const_cols = X_tr.columns[X_tr.nunique(dropna=True) <= 1]
    if len(const_cols):
        X_tr = X_tr.drop(columns=const_cols)
        X_val = X_val.drop(columns=const_cols, errors="ignore")

    # 3) winsorize léger (quantiles 0.1%–99.9%) fit sur TRAIN, appliqué à VAL
    q_lo = X_tr.quantile(0.001)
    q_hi = X_tr.quantile(0.999)
    X_tr = X_tr.clip(lower=q_lo, upper=q_hi, axis=1)
    X_val = X_val.clip(lower=q_lo, upper=q_hi, axis=1)

    # 4) imputation simple
    X_tr = X_tr.fillna(0.0)
    X_val = X_val.fillna(0.0)
    return X_tr, X_val


In [104]:
from sklearn.decomposition import PCA

X_decorr = PCA(n_components=0.999, random_state=0).fit_transform(X_train[features])


/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/decomposition/_pca.py:604: RuntimeWarning: divide by zero encountered in matmul
  C = X.T @ X
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/decomposition/_pca.py:604: RuntimeWarning: overflow encountered in matmul
  C = X.T @ X
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/decomposition/_pca.py:604: RuntimeWarning: invalid value encountered in matmul
  C = X.T @ X
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/alexandremasson/

In [96]:
X_decorr

array([[-5.03471402, -0.17146313,  0.81971908, ...,  0.19630338,
        -0.13352181,  0.26969362],
       [ 1.92642089,  0.01703463,  0.08878883, ..., -0.14823564,
        -0.34685473, -0.07424062],
       [-5.25594554,  0.01740076,  0.81389447, ...,  0.11914212,
         0.31738533, -0.13738461],
       ...,
       [-6.09097769,  0.39566588,  0.02774303, ..., -0.53358554,
        -0.25868754, -0.26224151],
       [ 2.64415999,  0.92329439, -1.68778958, ...,  0.8289901 ,
         1.65304382,  1.43463334],
       [-6.52750185, -2.21723351, -1.29078531, ..., -0.83838896,
        -0.79258616,  0.58786605]])

In [178]:
from sklearn import linear_model

acc = {i: [] for i in np.arange(0.1, 1, 0.1)}
for i in range(10):

    for alpha in np.arange(0.1, 1, 0.1):
        xt, xv , yt, yv = train_test_split(X_train[features], y_train, test_size=0.2)
        #xt, xv = clean_fold(xt, xv)

        new_ridge = linear_model.Ridge(alpha=alpha, fit_intercept=False)

        new_ridge.fit( xt.to_numpy(na_value=0),yt.to_numpy(na_value=0))

        preds_ridge = new_ridge.predict(xv)

        accuracy_ridge = (np.sign(preds_ridge) == np.sign(yv['target'].to_numpy())).mean()

        acc[alpha].append(accuracy_ridge)


for alpha in np.arange(0.1, 1, 0.1):
    print(f"Ridge accuracy {np.mean(acc[alpha])}")



/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=6.85975e-23): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", 

Ridge accuracy 0.523251685206247
Ridge accuracy 0.5209575855086133
Ridge accuracy 0.5225748287053732
Ridge accuracy 0.5218064301367583
Ridge accuracy 0.5221864684179865
Ridge accuracy 0.520380593081639
Ridge accuracy 0.5217398540874921
Ridge accuracy 0.5212349857138895
Ridge accuracy 0.5217481760936502


/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=6.44645e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but Ridge was fitted without feature names
  warnings.warn(
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/lin

In [160]:

new_ridge = linear_model.Ridge(alpha=0.5, fit_intercept=False)

new_ridge.fit( X_train[features].to_numpy(na_value=0),y_train.to_numpy(na_value=0))

preds_ridge = pd.DataFrame(new_ridge.predict(X_test[features].fillna(0).to_numpy(na_value=0)), index = X_test.index,columns=['target'])

(preds_ridge>0).astype(int).to_csv('data/preds_ridge_extended.csv')

/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=1.89604e-22): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", 

In [62]:
# elastic net 

from sklearn import linear_model

acc = {i: [] for i in np.arange(0.1, 1, 0.1)}
for i in range(10):

    for alpha in np.arange(0.1, 1, 0.1):
        xt, xv , yt, yv = train_test_split(X_train[features], y_train, test_size=0.2)
        #xt, xv = clean_fold(xt, xv)

        new_ridge = linear_model.ElasticNet(alpha=alpha, l1_ratio=0.1, fit_intercept=False)

        new_ridge.fit( xt.to_numpy(na_value=0),yt.to_numpy(na_value=0))

        preds_ridge = new_ridge.predict(xv)

        accuracy_ridge = (np.sign(preds_ridge) == np.sign(yv['target'].to_numpy())).mean()

        acc[alpha].append(accuracy_ridge)


for alpha in np.arange(0.1, 1, 0.1):
    print(f"Ridge accuracy {np.mean(acc[alpha])}")

/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but ElasticNet was fitted without feature names
  warnings.warn(
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountere

Ridge accuracy 0.49011623068601073
Ridge accuracy 0.49538960858831027
Ridge accuracy 0.4957696468695386
Ridge accuracy 0.4947016560792255
Ridge accuracy 0.4876640128713695
Ridge accuracy 0.48792199506227635
Ridge accuracy 0.4943854198452106
Ridge accuracy 0.49155316374934116
Ridge accuracy 0.4922910482953758


/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but ElasticNet was fitted without feature names
  warnings.warn(
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountere

# Models

In [26]:
from sklearn.linear_model import Ridge

class RidgeRegClassifier:
    def __init__(self, alpha=1e-2, fit_intercept=False):
        self.alpha = alpha
        self.model = Ridge(alpha=alpha)
    
    def fit(self, X, y):
        self.model.fit(X.to_numpy(na_value=0), y.to_numpy(na_value=0))
        return self
    
    def decision_function(self, X):

        preds= self.model.predict((X_test[features].fillna(0).to_numpy(na_value=0)))
        return (preds>0).astype(int)
        
        

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

base_models = [
    ("xgb_d4",
     xgb.XGBClassifier(
         n_estimators=4000, learning_rate=0.05, max_depth=4,
         subsample=0.9, colsample_bytree=0.7,
         objective="binary:logistic", eval_metric="logloss",
         tree_method="hist", random_state=10, early_stopping_rounds=100, n_jobs=-1
     )),
    ("xgb_d6",
     xgb.XGBClassifier(
         n_estimators=4000, learning_rate=0.05, max_depth=6,
         subsample=0.9, colsample_bytree=0.7,
         objective="binary:logistic", eval_metric="logloss",
         tree_method="hist", random_state=10, early_stopping_rounds=100, n_jobs=-1
     )),
    # ("rf",
    #  RandomForestClassifier(
    #      n_estimators=100, max_features="sqrt",
    #      min_samples_leaf=2, max_depth=20, n_jobs=-1, random_state=2)
    # ),
    ("et",
     ExtraTreesClassifier(
         n_estimators=300, max_features="sqrt",
         min_samples_leaf=2, n_jobs=-1, random_state=3)
    ),
    ("logit",
     Pipeline([("sc", StandardScaler()),
               ("clf", LogisticRegression(max_iter=2000, C=1.0, n_jobs=-1))])
    ),
    # ("ridge_clf",
    # RidgeRegClassifier()
    #  # proba via decision_function (dans ton code, utilise Calibrated si besoin)
    # ),
    # ("sgd_log",
    #  Pipeline([("sc", StandardScaler()),
    #            ("clf", SGDClassifier(loss="log_loss", alpha=1e-4, max_iter=3000, random_state=4))])
    # ),
    ("linsvc_cal",
     Pipeline([
         ("sc", StandardScaler()),
         ("clf", CalibratedClassifierCV(
             estimator=LinearSVC(C=1.0, random_state=5),
             cv=3,
             method="sigmoid"
      ))
  ])
),
    
]


In [180]:
import lightgbm as lgb
from catboost import CatBoostClassifier

base_models_ext = [
    ("lgbm",
     lgb.LGBMClassifier(
         n_estimators=4000, learning_rate=0.03,
         num_leaves=31, subsample=0.8, colsample_bytree=0.8,
         reg_lambda=1.0, random_state=7))
    ,
    ("cat",
     CatBoostClassifier(
         iterations=3000, learning_rate=0.03, depth=6,
         l2_leaf_reg=3.0, loss_function="Logloss",
         verbose=False, random_state=8))
]

In [181]:
base_models = base_models + base_models_ext

In [182]:
from sklearn.base import clone

# y binaire
y_bin = (y_train["target"] > 0).astype(int)

X = X_train[features].copy()


n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
train_dates = X_train['TS'].unique()

N = len(X_train)
L = len(base_models)
oof = np.full((N, L), np.nan, dtype=float)

fold_scores_per_model = {name: [] for name, _ in base_models}

# map index -> position pour remplir OOF proprement
pos_of = pd.Series(np.arange(N), index=X_train.index)

for i, (idx_tr_dates, idx_val_dates) in enumerate(kf.split(train_dates), 1):
    tr_dates = train_dates[idx_tr_dates]
    val_dates = train_dates[idx_val_dates]

    m_tr = X_train['TS'].isin(tr_dates)
    m_val = X_train['TS'].isin(val_dates)

    X_tr = X.loc[m_tr]
    y_tr = y_bin.loc[m_tr]
    X_val = X.loc[m_val]
    y_val = y_bin.loc[m_val]

    val_pos = pos_of.loc[X_val.index].to_numpy()

    fold_acc = []

    for j, (name, model_proto) in enumerate(base_models):
        model = clone(model_proto)
        fit_kwargs = dict(X=X_tr, y=y_tr)

        # early stopping pour XGB seulement
        if isinstance(model, xgb.XGBClassifier):
            fit_kwargs["eval_set"] = [(X_val, y_val)]
            fit_kwargs["verbose"] = False

        model.fit(**fit_kwargs)

        # probs validation
        p_val = (model.predict_proba(X_val)[:, 1]
                 if hasattr(model, "predict_proba")
                 else model.decision_function(X_val))

        # accuracy fold du modèle de base

        acc = accuracy_score(y_val, (p_val >= 0.5).astype(int))
        fold_scores_per_model[name].append(acc)

        fold_acc.append(acc)
        # stocke OOF
        oof[val_pos, j] = p_val

        print(f"Fold {i:02d} - {name}: Acc {acc*100:.2f}%")

    

    print(f"Fold {i:02d} done with acc {np.mean(fold_acc)*100:.2f}")

# Vérif OOF rempli
assert not np.isnan(oof).any(), "OOF incomplet (fuites ou split incohérent)."

# Report perfs moyennes des bases
for name in fold_scores_per_model:
    accs = np.array(fold_scores_per_model[name])
    print(f"[BASE] {name}: Acc mean={accs.mean()*100:.2f}%  std={accs.std()*100:.2f}%")

# Méta-learner (simple, efficace)
meta = LogisticRegression(max_iter=1000, class_weight=None, solver="lbfgs")
meta.fit(oof, y_bin.to_numpy())

# Score OOF du superlearner 
p_oof_meta = meta.predict_proba(oof)[:, 1]
acc_meta = accuracy_score(y_bin, (p_oof_meta >= 0.5).astype(int))
print(f"[META] Superlearner OOF Accuracy: {acc_meta*100:.2f}%")



Fold 01 - xgb_d4: Acc 52.31%
Fold 01 - xgb_d6: Acc 52.30%
Fold 01 - et: Acc 51.62%


/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:330: Run

Fold 01 - logit: Acc 52.29%


/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/calibration.py:867: RuntimeWarning: divide by zero encountered in matmul
  grad = np.asarray([-g @ F, -g.sum()], dtype=np.float64)
/Users/alexandremasson/Librar

Fold 01 - linsvc_cal: Acc 52.30%
[LightGBM] [Info] Number of positive: 81341, number of negative: 80834
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012850 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14535
[LightGBM] [Info] Number of data points in the train set: 162175, number of used features: 57
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501563 -> initscore=0.006253
[LightGBM] [Info] Start training from score 0.006253


KeyboardInterrupt: 

# Regression superlearner

## Models

In [14]:
from sklearn.linear_model import Lasso, ElasticNet, LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor


base_models = [

    # --- Boosting-based models (no scaling needed) ---
    ("xgb_4", xgb.XGBRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=0,
        n_jobs=-1,
        early_stopping_rounds = 50
    ), False),

    ("xgb_6", xgb.XGBRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=0,
        n_jobs=-1,
        early_stopping_rounds = 50
    ), False),

    (
        "lgb", lgb.LGBMRegressor(
            n_estimators=4000,
            learning_rate=0.03,
            num_leaves=31,
            feature_fraction=0.8,
            bagging_fraction=0.8,
            random_state=0,
            n_jobs=-1,
        ), False
    ),

    (
       "cat", CatBoostRegressor(
            iterations=3000,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=3,
            verbose=0,
            random_seed=0,
        ), False
    ),

    ("ridge", Ridge(alpha=2.0, random_state=0), True),
    ("lasso", Lasso(alpha=1e-3, random_state=0), True),
    ("enet", ElasticNet(alpha=2e-3, l1_ratio=0.4, random_state=0), True),
    ("ols", LinearRegression(), True),

    ("mlp", MLPRegressor(hidden_layer_sizes=(128,64),
                         activation="relu",
                         alpha=1e-3,
                         max_iter=400,
                         early_stopping=True,
                         random_state=0), True),

    ("svr", SVR(C=2.0, epsilon=1e-3, kernel="rbf"), True),
    ("knn", KNeighborsRegressor(n_neighbors=50, weights="distance"), True),

    ("tree", DecisionTreeRegressor(max_depth=5, random_state=0), False),
]


In [15]:
models = ["xgb_4", "xgb_6", "lgb", "ridge", "cat", "mlp"]
models = [x for x in base_models if x[0] in models]

In [20]:

y_cont = y_train["target"]
y_sign = (y_cont > 0).astype(int)
X = X_train.copy()

n_splits = 10
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
train_dates = X_train["TS"].unique()

N, L = len(X), len(models)
oof = np.full((N, L), np.nan)
fold_scores_per_model = {name: [] for name, _, _ in models}
pos_of = pd.Series(np.arange(N), index=X.index)

for i, (idx_tr_dates, idx_val_dates) in enumerate(kf.split(train_dates), 1):
    tr_dates, val_dates = train_dates[idx_tr_dates], train_dates[idx_val_dates]
    m_tr, m_val = X["TS"].isin(tr_dates), X["TS"].isin(val_dates)
    X_tr, y_tr = X.loc[m_tr, features], y_cont.loc[m_tr]
    X_val, y_val = X.loc[m_val, features], y_cont.loc[m_val]
    val_pos = pos_of.loc[X_val.index].to_numpy()

    fold_acc = []
    for j, (name, model_proto, need_scaling) in enumerate(models):
        model = clone(model_proto)
        
        if need_scaling:
            scaler = StandardScaler()
            X_tr_ = scaler.fit_transform(X_tr)
            X_val_ = scaler.transform(X_val)
        else:
            X_tr_, X_val_ = X_tr, X_val

        if name not in ['xgb_4', 'xgb_6']:
            model.fit(X_tr_, y_tr)
        else : 
            model.fit(X=X_tr_, y=y_tr, eval_set=[(X_val_, y_val)], verbose=False)


        p_val = model.predict(X_val_)  # prédiction continue
        acc = accuracy_score((y_val > 0), (p_val > 0))
        oof[val_pos, j] = p_val
        fold_scores_per_model[name].append(acc)
        fold_acc.append(acc)
        print(f"Fold {i:02d} - {name}: Acc {acc*100:.2f}%")

    print(f"Fold {i:02d} done. Mean acc={np.mean(fold_acc)*100:.2f}%")

assert not np.isnan(oof).any(), "OOF incomplet."

# performances moyennes
for name in fold_scores_per_model:
    accs = np.array(fold_scores_per_model[name])
    print(f"[BASE] {name}: mean={accs.mean()*100:.2f}% ±{accs.std()*100:.2f}%")

# méta-learner
meta = LogisticRegression(max_iter=1000, solver="lbfgs")
meta.fit(oof, y_sign)

p_oof_meta = meta.predict_proba(oof)[:, 1]
acc_meta = accuracy_score(y_sign, (p_oof_meta >= 0.5))
print(f"[META] Superlearner OOF Accuracy: {acc_meta*100:.2f}%")


Fold 01 - xgb_4: Acc 52.67%
Fold 01 - xgb_6: Acc 53.00%
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007510 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14075
[LightGBM] [Info] Number of data points in the train set: 162175, number of used features: 58
[LightGBM] [Info] Start training from score 0.000009
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fractio

/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexandremasson/Library/CloudStorage

Fold 01 - mlp: Acc 51.34%
Fold 01 done. Mean acc=52.20%
Fold 02 - xgb_4: Acc 52.76%
Fold 02 - xgb_6: Acc 52.87%
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009241 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14074
[LightGBM] [Info] Number of data points in the train set: 162175, number of used features: 58
[LightGBM] [Info] Start training from score 0.000011
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_by

/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexandremasson/Library/CloudStorage

Fold 02 - mlp: Acc 50.86%
Fold 02 done. Mean acc=51.99%
Fold 03 - xgb_4: Acc 50.40%
Fold 03 - xgb_6: Acc 50.45%
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008167 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14074
[LightGBM] [Info] Number of data points in the train set: 162175, number of used features: 58
[LightGBM] [Info] Start training from score 0.000013
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_by

...

/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexandremasson/Library/CloudStorage

Fold 09 - mlp: Acc 51.37%
Fold 09 done. Mean acc=51.81%
Fold 10 - xgb_4: Acc 53.89%
Fold 10 - xgb_6: Acc 53.70%
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009016 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14074
[LightGBM] [Info] Number of data points in the train set: 162240, number of used features: 58
[LightGBM] [Info] Start training from score 0.000012
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_by

/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexandremasson/Library/CloudStorage

Fold 10 - mlp: Acc 52.59%
Fold 10 done. Mean acc=52.90%
[BASE] xgb_4: mean=52.08% ±0.94%
[BASE] xgb_6: mean=52.00% ±1.09%
[BASE] lgb: mean=51.34% ±0.58%
[BASE] cat: mean=51.69% ±0.52%
[BASE] ridge: mean=51.89% ±0.98%
[BASE] mlp: mean=51.15% ±0.72%
[META] Superlearner OOF Accuracy: 50.28%


/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/alexandremasson/Library

In [22]:

Z = StandardScaler().fit_transform(oof)              # center/scale chaque colonne OOF
meta = LogisticRegression(C=0.2, max_iter=2000)      # L2 plus forte
meta.fit(Z, y_sign)
p_meta = meta.predict_proba(Z)[:,1]
acc = accuracy_score(y_sign, p_meta >= 0.5)
print("META zscore+logit:", acc*100)

META zscore+logit: 52.30269910399734


/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

In [24]:
from sklearn.isotonic import IsotonicRegression

P = np.zeros_like(oof)
for j in range(oof.shape[1]):
    ir = IsotonicRegression(out_of_bounds="clip")
    ir.fit(oof[:, j], y_sign)          # map score_j -> prob_j
    P[:, j] = ir.transform(oof[:, j])

p_blend = P.mean(axis=1)               # simple average des probas calibrées
acc = accuracy_score(y_sign, p_blend >= 0.5)
print("BLEND isotonic-avg:", acc*100)

BLEND isotonic-avg: 52.24000665760493


In [25]:
import numpy as np
from scipy.stats import rankdata

R = np.column_stack([rankdata(oof[:,j]) / (len(oof)+1) for j in range(oof.shape[1])])
r_blend = R.mean(axis=1)
acc = accuracy_score(y_sign, r_blend >= 0.5)         # seuil 0.5 sur ranks moyens
print("BLEND rank-avg:", acc*100)


BLEND rank-avg: 52.09353934921912


In [26]:
# --- calibration isotone par base (fit sur OOF vs y_sign) ---
isos = []
P_oof = np.zeros_like(oof)
for j, (name, _, _) in enumerate(models):
    ir = IsotonicRegression(out_of_bounds="clip")
    ir.fit(oof[:, j], y_sign.values)     # map score_j -> prob_j
    P_oof[:, j] = ir.transform(oof[:, j])
    isos.append(ir)

# --- blend & sélection seuil sur OOF ---
p_blend_oof = P_oof.mean(axis=1)

thr_grid = np.linspace(0.3, 0.7, 401)     # grille compacte et sûre
accs = [(thr, accuracy_score(y_sign, p_blend_oof >= thr)) for thr in thr_grid]
best_thr, best_acc = max(accs, key=lambda t: t[1])
print(f"[OOF] Isotonic blend acc={best_acc*100:.2f}%  @thr={best_thr:.3f}")
#[OOF] Isotonic blend acc=52.27%  @thr=0.503


[OOF] Isotonic blend acc=52.29%  @thr=0.504


In [48]:
X = X_train[features]
X_te = X_test[features]
# --- REFIT final de chaque base sur TOUT le train + prédiction test ---
P_test_cols = []
for (name, model_proto, need_scaling), ir in zip(models, isos):
    

    if name not in ['xgb_4', 'xgb_6']:
        model = clone(model_proto)

        if need_scaling:
            sc = StandardScaler()
            X_tr_full = sc.fit_transform(X)
            X_te_full = sc.transform(X_te)
        else:
            X_tr_full, X_te_full = X, X_te

    elif name == 'xgb_4' :

        model = xgb.XGBRegressor(
        n_estimators=350,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=0,
        n_jobs=-1)

    elif name == 'xgb_6' :
        model = xgb.XGBRegressor(
        n_estimators=350,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        random_state=0,
        n_jobs=-1)

    model.fit(X_tr_full, y_cont)


    s_test = model.predict(X_te_full)   
    p_test = ir.transform(s_test)          
    P_test_cols.append(p_test)

p_test_blend = np.column_stack(P_test_cols).mean(axis=1)  

pred_test = (p_test_blend >= best_thr).astype(int)

# --- sorties utiles ---
submission_proba = pd.Series(p_test_blend, index=X_test.index, name="proba")
submission_pred  = pd.Series(pred_test,   index=X_test.index, name="prediction")

print("Test preds ready:",
      f"mean_proba={submission_proba.mean():.3f}, "
      f"share_pos={submission_pred.mean():.3f}")

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010074 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14075
[LightGBM] [Info] Number of data points in the train set: 180245, number of used features: 58
[LightGBM] [Info] Start training from score 0.000012
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, 

/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexandremasson/Library/CloudStorage

Test preds ready: mean_proba=0.504, share_pos=0.473


/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/alexandremasson/Library/CloudStorage/OneDrive-Personnel/ALEX/Tryhard/Challenge data/QRT_kaggle/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [51]:
sub = pd.DataFrame(submission_pred)

In [53]:
sub.to_csv("data/supermodel_try.csv")